# Causal Uplift Modeling on Criteo Uplift v2.1

This notebook is the front door to the project. It lays out the decision problem, why it is not the same problem as predicting who will convert, how the study is designed to answer it, and what the rest of the notebooks in this series do. It runs no analysis of its own: no data is loaded, no model is trained, and no result is reported here.

## 1. The decision problem

An advertiser can choose, for each person, whether to show them an ad. Showing an ad costs money and attention; not showing one risks a lost sale. The question worth answering is not *who is likely to buy* but **whose purchase is caused by seeing the ad** — because those are frequently different people.

That distinction is the entire subject of this project: given a person's pre-treatment characteristics, rank or select them by how much showing the ad **changes** their chance of converting, using [CRITEO-UPLIFTv2.1](https://ailab.criteo.com/criteo-uplift-prediction-dataset/), a large-scale randomized-assignment advertising dataset released for exactly this kind of research.

## 2. Why predicting who converts is not enough

The familiar tool for this kind of targeting is a response model: estimate $P(Y=1 \mid X)$, the probability of conversion given a person's features, and target the people with the highest predicted probability.

That model answers a different question than the one that matters for targeting. Consider two people with the same predicted conversion probability:

- One was going to convert whether or not they saw the ad — showing them the ad spends budget on a sale that would have happened anyway.
- The other converts only if shown the ad — showing them the ad causes an incremental sale.

A response model cannot tell these two people apart, because it never compares the same person's outcome with and without treatment. Ranking by response probability and ranking by incremental effect can disagree almost completely, and a targeting policy built on the wrong ranking spends budget on people it cannot influence.

## 3. Uplift and CATE, in one line

Uplift modeling (equivalently, heterogeneous treatment effect or CATE — conditional average treatment effect — modeling) targets

$$\tau(x) = E[Y(1) - Y(0) \mid X = x],$$

the expected change in outcome caused by treatment, for people who share features $x$. It is a **contrast**, not a level: a person with a low absolute conversion probability can still have a large uplift, and a person with a high conversion probability can have an uplift near zero. Targeting by $\tau(x)$ instead of $P(Y=1\mid X=x)$ is the entire point of this project.

## 4. Potential outcomes, and why $\tau(x)$ is estimated rather than observed

For a given row $i$, write $Y_i(1)$ for the conversion outcome that would occur if that row were treated, and $Y_i(0)$ for the outcome if it were not. The individual treatment effect $Y_i(1) - Y_i(0)$ is a well-defined quantity, but it is never observed directly: a row is either treated or not, so only one of $Y_i(1)$ and $Y_i(0)$ is ever realized for it. This is the fundamental problem of causal inference, and it is why the project estimates $\tau(x)$ — an average over people who share features $x$ — rather than reading off individual effects from the data.

A fitted model's output, $\hat\tau(x) = \hat\mu_1(x) - \hat\mu_0(x)$, is therefore an **estimate of a conditional average contrast**. Throughout this project it is called predicted uplift or estimated CATE — never a true individual treatment effect, and never validated against one, because no such ground truth exists on this dataset.

## 5. Dataset and variable roles

CRITEO-UPLIFTv2.1 contains roughly 14 million released rows from a randomized advertising experiment. Each row is one released observation — not an assumed unique person, since the public schema carries no durable user identifier.

| Role | Field(s) | Notes |
|---|---|---|
| Covariates `X` | `f0` through `f11`, in that order | The only model inputs. Anonymized: Criteo does not publish what they represent, and this project does not invent a meaning for them. |
| Treatment `T` | `treatment` | Binary randomized assignment (1 = ad assigned, 0 = not). This is *assignment*, used throughout as the treatment — never realized exposure. |
| Primary outcome `Y` | `conversion` | Binary. The outcome the primary analysis is built around. |
| Secondary outcome | `visit` | Binary, evaluated separately as a robustness check. Never a model input, never a substitute for `conversion`, never used to filter rows or select a model. |
| Audit-only | `exposure` | Whether the ad was actually seen. This happens *after* assignment and depends on the person's own behavior, so using it as a feature or restricting to exposed rows would replace a randomized comparison with a self-selected one. Diagnostic only — never a feature, never a filter. |
| Row identity | `_source_row_id` | A zero-based ordinal used for provenance and alignment across the pipeline. Not a person identifier, and never a model input. |

Every released row that passes basic integrity checks (correct schema, valid `X`/`T`/`Y`, a stable row identity, no forbidden variable in `X`) is kept. Rows are not excluded for being rare, extreme, or repeated — removing them would quietly change which population the results describe.

## 6. Target estimand

The primary quantity this project ranks by is the assignment (intention-to-treat) CATE,

$$\tau(x) = E[Y(1) - Y(0) \mid X=x],$$

estimated over the eligible released population described above. The population average, $ATE = E[Y(1)-Y(0)]$, is reported as a secondary summary, not the primary target — the project's purpose is ranking and targeting, which needs the conditional version.

Using *assignment* rather than realized exposure as the treatment keeps the randomized comparison intact: assignment was decided before any of a person's own downstream behavior, while exposure was not.

## 7. Identification assumptions and limitations

Estimating a causal effect from this data — rather than a purely descriptive contrast — relies on assumptions that the data alone cannot prove:

- **Exchangeability.** Treatment assignment is independent of potential outcomes. This is supported by the dataset's randomized design, but that design claim is not independently re-derived here from a primary source, and no empirical balance check can *prove* randomization — balance is consistent with it at best.
- **Positivity/overlap.** Both assignment arms must be represented in the relevant regions of feature space. Checked empirically; a region with no rows on one arm cannot support an uplift estimate there.
- **Pre-treatment covariates.** Every field in `X` must precede assignment and be unaffected by it. Timing is not independently source-verified for this anonymized release.
- **No interference.** One row's assignment must not affect another row's outcome — plausible for individually targeted ads, but not verifiable without a network or household identifier, which this schema does not have.
- **Consistent, well-defined outcomes.** `conversion` must mean the same thing for every row over a comparable window.

None of these are established by descriptive statistics or balance diagnostics; they are assumptions the analysis depends on and states plainly rather than treats as proven. The rows in this release are also not assumed to represent unique users, other campaigns, other platforms, other time periods, or populations outside the observed covariate support — conclusions are scoped to the released population under its observed assignment.

## 8. Model portfolio

All methods are compared under one shared population, split, and metric protocol; none is declared the winner in advance.

| Method | Role |
|---|---|
| Theoretical/seeded random ranking | Sanity and expected-random reference — what a ranking with no information would achieve |
| Response model (LightGBM) | Predicts $P(Y=1\mid X)$ as a targeting reference; explicitly not a causal estimator |
| T-Learner | Two outcome models, one per arm; $\hat\tau(x)=\hat\mu_1(x)-\hat\mu_0(x)$. The primary causal baseline |
| X-Learner | Cross-fitted signed pseudo-effect regressions, designed to exploit the arms' unequal sizes |
| Causal Forest | A forest built to target treatment-effect heterogeneity directly, rather than fitting two separate outcome models |
| DR-Learner | An orthogonalized, doubly-robust comparator; included only if it clears its own promotion checks in development — otherwise it is left out rather than forced in |

A single-model (S-Learner) baseline is not part of the active comparison. LightGBM is the shared base-learner family for the response, outcome, and effect stages.

## 9. Evaluation framework

Uplift-ranking quality, not response accuracy, is what selects a model:

- **Primary:** Qini above the theoretical random reference — how much more cumulative incremental outcome the ranking captures than an uninformed ranking would.
- **Secondary:** raw Qini/AUUC (the full curve), uplift at fixed top-K budgets, and estimated incremental conversions at those budgets.
- **Diagnostic only:** the response model's ROC-AUC, average precision, and log loss describe how well it predicts conversion — they do not measure uplift-ranking quality and never select a causal winner.
- **Uncertainty:** 500 paired, treatment-arm-stratified bootstrap draws on fixed predictions, reported alongside every comparison.

Because only one potential outcome is ever observed per row, this project does not report PEHE (error against a true individual effect) on the real dataset — no such ground truth exists here. Any such comparison would require a synthetic or semi-synthetic experiment with known effects, which is out of scope.

## 10. Train / validation / held-out protocol

The data is split once into 70% training, 15% validation, and 15% held-out test, stratified jointly on treatment and outcome so every partition has a comparable mix of arms and outcomes.

- **Training** fits candidate models.
- **Validation** drives every development decision: early stopping, hyperparameters, method comparison, and model selection. Reusing it repeatedly during development is expected — it is not a stand-in for final evaluation.
- **Held-out test** is opened exactly once, after every development and selection decision is frozen, to produce the one evaluation this project treats as final. Nothing about feature handling, thresholds, model choice, or hyperparameters may be adjusted using held-out information, and a disappointing held-out result is reported as-is rather than used to justify a second attempt on the same test set.

This separation exists so that the final number reported is a genuine estimate of out-of-sample performance, not a number the modeling process was indirectly optimized to produce.

## 11. Notebook roadmap

The study proceeds through a fixed sequence of notebooks, each building on the artifacts of the ones before it:

| Notebook | Covers |
|---|---|
| `00_project_overview.ipynb` | This notebook: problem, design, and roadmap |
| `01_data_feasibility.ipynb` | Full-data integrity and computational feasibility |
| `02_eda_split_preprocessing.ipynb` | Exploratory analysis, the frozen train/validation/held-out split, and preprocessing |
| `03_metrics_and_meta_learners.ipynb` | Uplift metrics, the random/response references, T-Learner, and X-Learner |
| `04_causal_forest.ipynb` | Causal Forest verification and training |
| `05_validation_uncertainty.ipynb` | Common validation comparison, decile/segment analysis, and bootstrap uncertainty |
| `06_heldout_evaluation.ipynb` | The pre-test freeze and the one-shot held-out evaluation |
| `07_final_story.ipynb` | The final comparison, limitations, and defense of the conclusion |

Each notebook exposes its question, method, and result directly — not as a wrapper around opaque helper functions — so it can be read and checked on its own.

## 12. What would make the conclusion valid

A defensible final claim from this project has to satisfy all of the following:

- It ranks by, or reports, $\tau(x)$ or one of its accepted summaries — never response probability presented as if it were a causal ranking.
- It compares models on the same held-out rows, using the metrics fixed in section 9, with a declared winner determined by the primary metric and not overturned by a secondary one.
- It reports uncertainty alongside every headline number, from the frozen bootstrap procedure.
- It states its scope as the eligible released CRITEO-UPLIFTv2.1 population under its observed covariate support — not users in general, other campaigns, or other platforms.
- It never treats predicted uplift as a true individual treatment effect, and never reports PEHE against one on this dataset.
- It reports an unfavorable or null result exactly as it would report a favorable one.

A result that violates any of these is not a valid conclusion for this project, regardless of how favorable it looks.